# 06 · Main experiment: scaled sweep on a fine temperature grid (design doc §8.8.11)

**How to run**: pick a Colab server with an **A100 GPU** and click "Run All"; about 2.5 to 3 hours. If the connection drops, click "Run All" again; finished parts are skipped. Results go to `DATA_DIR/06_scaled_sweep/`.

**What it computes**: 1,000 MAGE human texts (seven domains), 300 of which serve as continuation prompts. Three generators (Qwen2.5-3B, phi-2, SmolLM2-1.7B) continue them on a 0.02 temperature grid around the crossing, 8,100 texts in total; GPT-2 XL and each generator itself score them. Generation is batched (rules in paper §4.1), which needs an A100.

In [ ]:
# 1. Check that the runtime has an A100.
import torch
assert torch.cuda.is_available(), "No GPU: switch the Colab server to an A100 GPU"
name = torch.cuda.get_device_name(0); mem = torch.cuda.get_device_properties(0).total_memory / 2**30
print(name, f"{mem:.0f} GB")
assert mem > 30, f"This {name} has only {mem:.0f} GB; batched generation would be slow. Choose an A100 GPU."

In [ ]:
# 2. Mount Google Drive and switch to the data folder.
import os
from google.colab import drive
drive.mount("/content/drive")
DATA_DIR = "/content/drive/MyDrive/curvature-margin/data"  # folder with the layout in data/README.md
%cd $DATA_DIR
os.makedirs("06_scaled_sweep", exist_ok=True)
os.environ["HF_HOME"] = "/content/hf"

In [ ]:
# 3. Install packages.
%pip install -q -U transformers huggingface-hub accelerate
import json, gc, glob, shutil, time, traceback
import numpy as np, pandas as pd

In [ ]:
# 4. Draw 1,000 human texts: about 143 from each of seven domains, 120 to 300 English words, none of the earlier 100, seed 1.
DOMAINS = ["xsum", "eli5", "yelp", "wp", "sci_gen", "cmv", "squad"]
if not os.path.exists("06_scaled_sweep/humans_1000.json"):
    if not os.path.exists("/content/test.csv"):
        !curl -sSL --retry 5 -o /content/test.csv https://huggingface.co/datasets/yaful/MAGE/resolve/main/test.csv
    d = pd.read_csv("/content/test.csv").dropna(subset=["text"]).drop_duplicates("text")
    old = {h["text"] for h in json.load(open("01_pilot_decoding/humans.json"))}
    d = d[d.src.isin([f"{x}_human" for x in DOMAINS]) & ~d.text.isin(old)]
    words = d.text.str.split().str.len(); d = d[(words >= 120) & (words <= 300)]
    per = {dom: 143 for dom in DOMAINS}; per["xsum"] = 142   # 1,000 texts
    pick = pd.concat([d[d.src == f"{dom}_human"].sample(n, random_state=1) for dom, n in per.items()])
    pick = pick.sample(frac=1, random_state=1)              # shuffle so that the first 300 prompts keep similar domain shares
    rows = [dict(id=i, domain=s.replace("_human", ""), text=t) for i, (s, t) in enumerate(zip(pick.src, pick.text))]
    json.dump(rows, open("06_scaled_sweep/humans_1000.json", "w"), ensure_ascii=False)
humans = json.load(open("06_scaled_sweep/humans_1000.json"))
sources = humans[:300]
print(len(humans), "human texts; prompts:", len(sources), pd.Series([h["domain"] for h in sources]).value_counts().to_dict())

In [ ]:
# 5. Batched continuations: three generators, each on its own temperature grid. Rules (paper §4.1): first 30 tokens, plain softmax sampling,
#    temperature, top-p 1.0, top-k off, special and end-of-text tokens banned, exact length. Each (model, temperature) is saved when done.
from transformers import AutoTokenizer, AutoModelForCausalLM
GENS = {"Qwen/Qwen2.5-3B": [0.6, 0.8, 0.9, 0.92, 0.94, 0.96, 0.98, 1.0, 1.02, 1.04, 1.06, 1.1, 1.2],
        "microsoft/phi-2": [0.8, 0.9, 0.94, 0.98, 1.0, 1.04, 1.1],
        "HuggingFaceTB/SmolLM2-1.7B": [0.8, 0.9, 0.94, 0.98, 1.0, 1.04, 1.1]}
PREFIX, MAX_TOK, BS, SEED = 30, 512, 24, 42

def prepare(tok, texts):
    items = []
    for h in texts:
        ids = tok(h["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) < PREFIX + 20:
            continue
        items.append(dict(id=h["id"], domain=h["domain"], prefix_ids=ids[:PREFIX], target=len(ids) - PREFIX,
                          human=tok.decode(ids[PREFIX:], clean_up_tokenization_spaces=False),
                          prefix=tok.decode(ids[:PREFIX], clean_up_tokenization_spaces=False)))
    return items

@torch.no_grad()
def generate_batch(model, tok, batch, temp, special):
    pad = tok.pad_token_id
    L = max(len(b["prefix_ids"]) for b in batch); new = max(b["target"] for b in batch)
    x = torch.full((len(batch), L), pad); att = torch.zeros((len(batch), L), dtype=torch.long)
    for i, b in enumerate(batch):
        x[i, L - len(b["prefix_ids"]):] = torch.tensor(b["prefix_ids"]); att[i, L - len(b["prefix_ids"]):] = 1
    out = model.generate(input_ids=x.cuda(), attention_mask=att.cuda(), do_sample=True, temperature=temp, top_p=1.0, top_k=0,
                         max_new_tokens=new, min_new_tokens=new, suppress_tokens=special, pad_token_id=pad,
                         repetition_penalty=1.0)
    gen = out[:, L:].cpu()
    res = [gen[i, :b["target"]].tolist() for i, b in enumerate(batch)]
    for b, ids in zip(batch, res):   # self-check: exact length, no special tokens (continuation rules, paper §4.1)
        assert len(ids) == b["target"], ("wrong length", len(ids), b["target"])
        assert not (set(ids) & set(special)), "a special token was generated"
    return res

for gen_name, temps in GENS.items():
    tag = gen_name.split("/")[1]
    todo = [t for t in temps if not os.path.exists(f"06_scaled_sweep/gen_{tag}_T{t}.json")]
    if not todo:
        print("Already done, skipping:", tag); continue
    tok = AutoTokenizer.from_pretrained(gen_name); tok.padding_side = "left"
    if tok.pad_token_id is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(gen_name, dtype=torch.bfloat16).cuda().eval()
    special = sorted(set(tok.all_special_ids) | {tok.eos_token_id} | set(np.atleast_1d(model.generation_config.eos_token_id).tolist() if model.generation_config.eos_token_id is not None else []))
    items = sorted(prepare(tok, sources), key=lambda b: b["target"])
    for temp in todo:
        torch.manual_seed(SEED); t0 = time.time(); rows = []
        for s in range(0, len(items), BS):
            batch = items[s:s + BS]
            for b, ids in zip(batch, generate_batch(model, tok, batch, temp, special)):
                rows.append(dict(id=b["id"], domain=b["domain"], model=gen_name, decoding=f"T{temp}", prefix=b["prefix"],
                                 ai=tok.decode(ids, clean_up_tokenization_spaces=False), human=b["human"], n_tokens=len(ids)))
        json.dump(rows, open(f"06_scaled_sweep/gen_{tag}_T{temp}.json", "w"), ensure_ascii=False)
        print(f"{tag} T={temp}: {len(rows)} texts, {(time.time() - t0) / 60:.1f} min", flush=True)
    # save the 1,000 human texts split with this model's prefix tokenization, as the reference set for scoring
    hpath = f"06_scaled_sweep/humans_{tag}.json"
    if not os.path.exists(hpath):
        json.dump([dict(id=b["id"], domain=b["domain"], prefix=b["prefix"], human=b["human"]) for b in prepare(tok, humans)],
                  open(hpath, "w"), ensure_ascii=False)
    del model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/hf/models--{gen_name.replace('/', '--')}", ignore_errors=True)

In [ ]:
# 6. Batched scoring: GPT-2 XL scores all texts; each generator scores its own texts and the human texts. Continuation only; four values per position.
@torch.no_grad()
def score_records(scorer_name, recs, out_path, bs=16):
    if os.path.exists(out_path):
        print("Already done, skipping:", out_path); return
    tok = AutoTokenizer.from_pretrained(scorer_name); tok.padding_side = "right"
    pad = tok.pad_token_id if tok.pad_token_id is not None else (tok.eos_token_id or 0)
    model = AutoModelForCausalLM.from_pretrained(scorer_name, dtype=torch.bfloat16).cuda().eval()
    enc = []
    for r in recs:
        k = len(tok(r["prefix"])["input_ids"])
        ids = tok(r["prefix"] + r["text"], truncation=True, max_length=MAX_TOK)["input_ids"]
        if len(ids) - k >= 10:
            enc.append((r, k, ids))
    enc.sort(key=lambda e: len(e[2]))
    out, t0 = [], time.time()
    for s in range(0, len(enc), bs):
        chunk = enc[s:s + bs]; L = max(len(e[2]) for e in chunk)
        x = torch.full((len(chunk), L), pad); att = torch.zeros((len(chunk), L), dtype=torch.long)
        for i, (_, _, ids) in enumerate(chunk):
            x[i, :len(ids)] = torch.tensor(ids); att[i, :len(ids)] = 1
        logits = model(input_ids=x.cuda(), attention_mask=att.cuda()).logits
        for i, (r, k, ids) in enumerate(chunk):
            n = len(ids)
            lp = torch.log_softmax(logits[i, :n - 1].float(), -1)
            tgt = torch.tensor(ids[1:], device=lp.device)
            p = lp.exp(); e1 = (p * lp).sum(-1)
            arr = torch.stack([-lp.gather(1, tgt[:, None])[:, 0], -e1, (p * lp * lp).sum(-1) - e1 ** 2,
                               p.topk(10, -1).values.sum(-1)]).cpu().numpy().astype(np.float32)[:, k - 1:]
            out.append(dict(id=r["id"], domain=r["domain"], model=r["model"], decoding=r["decoding"], who=r["who"],
                            scorer=scorer_name, arr=arr))
        if (s // bs) % 50 == 0:
            print(f"  {scorer_name}: {s}/{len(enc)} {(time.time() - t0) / 60:.1f} min", flush=True)
    pd.DataFrame(out).to_pickle(out_path)
    del model; gc.collect(); torch.cuda.empty_cache()
    shutil.rmtree(f"/content/hf/models--{scorer_name.replace('/', '--')}", ignore_errors=True)

def records_for(gen_name):
    tag = gen_name.split("/")[1]; recs = []
    for f in sorted(glob.glob(f"06_scaled_sweep/gen_{tag}_T*.json")):
        for r in json.load(open(f)):
            recs.append(dict(id=r["id"], domain=r["domain"], model=gen_name, decoding=r["decoding"], who="ai", prefix=r["prefix"], text=r["ai"]))
    for r in json.load(open(f"06_scaled_sweep/humans_{tag}.json")):
        recs.append(dict(id=r["id"], domain=r["domain"], model=gen_name, decoding="human", who="human", prefix=r["prefix"], text=r["human"]))
    return recs

for gen_name in GENS:
    tag = gen_name.split("/")[1]
    score_records("gpt2-xl", records_for(gen_name), f"06_scaled_sweep/scores_gpt2-xl_{tag}.pkl")
for gen_name in GENS:
    tag = gen_name.split("/")[1]
    score_records(gen_name, records_for(gen_name), f"06_scaled_sweep/scores_self_{tag}.pkl")

In [ ]:
# 7. Quick summary: margin and AUROC per (generator, scorer, temperature) against the 1,000 human texts with the same split. The full analysis is scripts/e1_scaled_sweep.py.
from sklearn.metrics import roc_auc_score
rows = []
for f in sorted(glob.glob("06_scaled_sweep/scores_*.pkl")):
    s = pd.read_pickle(f)
    s["surprisal"] = [a[0].mean() for a in s.arr]
    s["fdg"] = [(a[1].sum() - a[0].sum()) / np.sqrt(a[2].sum()) for a in s.arr]
    h = s[s.who == "human"]
    for dec, g in s[s.who == "ai"].groupby("decoding"):
        y = np.r_[np.ones(len(g)), np.zeros(len(h))]
        rows.append(dict(file=f.split("/")[-1], temperature=float(dec[1:]), n_ai=len(g), surprisal_margin=h.surprisal.mean() - g.surprisal.mean(),
                         AUROC=roc_auc_score(y, np.r_[g.fdg.values, h.fdg.values])))
table = pd.DataFrame(rows).sort_values(["file", "temperature"]); table.to_csv("06_scaled_sweep/summary.csv", index=False)
print(table.round(3).to_string(index=False))

In [ ]:
# 8. Flush unsynced files back to Google Drive.
%cd /content
drive.flush_and_unmount()
print("Done: results are in DATA_DIR/06_scaled_sweep/")